# ¿Qué es un LLM? — Notebook interactivo

Exploramos los conceptos fundamentales de los Large Language Models:
tokens, temperatura, probabilidades y cómo afectan a la generación de texto.

In [ ]:
import anthropic, os, json
import matplotlib.pyplot as plt
import numpy as np

client = anthropic.Anthropic()
MODELO = 'claude-haiku-4-5-20251001'
print('Cliente listo')

## 1. Tokens — la unidad básica de un LLM

In [ ]:
# Contar tokens con la API de Anthropic
def contar_tokens(texto: str) -> int:
    resp = client.messages.count_tokens(
        model=MODELO,
        messages=[{'role': 'user', 'content': texto}]
    )
    return resp.input_tokens

ejemplos = [
    'Hola',
    'Hello',
    'Inteligencia artificial',
    'Artificial intelligence',
    'El proceso de tokenización divide el texto en unidades mínimas llamadas tokens.',
    'The tokenization process splits text into minimum units called tokens.',
    'αβγδεζηθ',  # caracteres griegos
    '你好世界',    # chino
]

print(f'{"Texto":<55} {"Tokens":>6} {"Ratio tok/palabra":>18}')
print('-' * 82)
for texto in ejemplos:
    tokens = contar_tokens(texto)
    palabras = max(len(texto.split()), 1)
    ratio = tokens / palabras
    print(f'{texto:<55} {tokens:>6} {ratio:>18.2f}')

## 2. Temperatura — creatividad vs determinismo

In [ ]:
def generar_con_temperatura(prompt: str, temperatura: float, n_muestras: int = 3) -> list[str]:
    respuestas = []
    for _ in range(n_muestras):
        resp = client.messages.create(
            model=MODELO,
            max_tokens=80,
            temperature=temperatura,
            messages=[{'role': 'user', 'content': prompt}]
        )
        respuestas.append(resp.content[0].text.strip())
    return respuestas

prompt_test = 'Completa esta frase con exactamente 5 palabras: "La inteligencia artificial es"'

for temp in [0.0, 0.5, 1.0]:
    respuestas = generar_con_temperatura(prompt_test, temp)
    variedad = len(set(respuestas))  # cuántas únicas
    print(f'\nTemperatura {temp} (variedad: {variedad}/3):')
    for r in respuestas:
        print(f'  → {r}')

## 3. El LLM como completador de texto

In [ ]:
# Un LLM predice el siguiente token más probable dada una secuencia
# Esto explica por qué el prompt importa tanto

prompts_equivalentes = [
    'La capital de Francia es',
    'Francia. Capital:',
    'Q: ¿Cuál es la capital de Francia? A:',
    'Escribe la capital de Francia y nada más:',
]

print('Mismo conocimiento, diferentes prompts → diferentes respuestas:')
for prompt in prompts_equivalentes:
    resp = client.messages.create(
        model=MODELO, max_tokens=20, temperature=0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    print(f'  Prompt: "{prompt}"')
    print(f'  Output: "{resp.content[0].text.strip()}"\n')

## 4. Ventana de contexto

In [ ]:
# La ventana de contexto es el máximo de tokens que el modelo puede "ver" a la vez
# Todo lo que está fuera de la ventana, el modelo no lo recuerda

VENTANAS_CONTEXTO = {
    'claude-haiku-4-5-20251001': 200_000,
    'claude-sonnet-4-6':         200_000,
    'gpt-4o':                    128_000,
    'gpt-4o-mini':               128_000,
    'gemini-1.5-pro':          1_000_000,
    'gemini-1.5-flash':        1_000_000,
    'mistral-small':             32_000,
    'mistral-medium':           128_000,
    'llama-3.1-70b':            128_000,
}

PAGINAS_A4_POR_TOKEN = 1 / 500  # aprox 500 tokens por página A4

print(f'{"Modelo":<30} {"Tokens":>12} {"Páginas A4":>12} {"Horas de audio":>15}')
print('-' * 72)
for modelo, tokens in sorted(VENTANAS_CONTEXTO.items(), key=lambda x: x[1]):
    paginas = tokens * PAGINAS_A4_POR_TOKEN
    horas_audio = tokens / 1500 / 60  # ~1500 tokens por minuto de audio transcrito
    print(f'{modelo:<30} {tokens:>12,} {paginas:>12,.0f} {horas_audio:>15.1f}h')

## 5. Alucinaciones — el talón de Aquiles

In [ ]:
# Demostración de alucinaciones y cómo mitigarlas

def preguntar(pregunta: str, con_instruccion: bool = False) -> str:
    system = '' if not con_instruccion else (
        'Responde SOLO con información que sepas con certeza. '
        'Si no estás seguro, di explícitamente "No tengo información fiable sobre esto".'
    )
    resp = client.messages.create(
        model=MODELO, max_tokens=150, temperature=0,
        system=system,
        messages=[{'role': 'user', 'content': pregunta}]
    )
    return resp.content[0].text.strip()

# Preguntas diseñadas para provocar alucinaciones
preguntas_trampa = [
    '¿En qué año publicó Borges su novela "La ciudad de los espejos infinitos"?',
    '¿Quién ganó el Nobel de Física en 2025?',
]

for pregunta in preguntas_trampa:
    print(f'Pregunta: {pregunta}')
    print(f'Sin instrucción: {preguntar(pregunta)}')
    print(f'Con instrucción: {preguntar(pregunta, con_instruccion=True)}')
    print()

## 6. LLM como clasificador, extractor y razonador

In [ ]:
TEXTO = """
Ayer asistí a la conferencia de IA en Madrid. El ponente, Dr. García, explicó
cómo los LLMs están transformando el sector legal. Me pareció brillante aunque
los precios de las herramientas siguen siendo altos para las PYMES. En general,
una jornada muy productiva que recomendaría a cualquier profesional del sector.
"""

def analizar_texto(texto: str) -> dict:
    resp = client.messages.create(
        model=MODELO, max_tokens=300, temperature=0,
        messages=[{'role': 'user', 'content': f'''Analiza este texto. Responde en JSON:
{{
  "sentimiento": "positivo|negativo|neutro",
  "entidades": {{"personas": [], "lugares": [], "organizaciones": []}},
  "temas": ["lista de temas principales"],
  "intención_autor": "descripción en 1 frase",
  "puntuacion_credibilidad": 0
}}

TEXTO: {texto}'''}]
    )
    texto_resp = resp.content[0].text
    if '```' in texto_resp:
        texto_resp = texto_resp.split('```')[1].replace('json', '').strip()
    return json.loads(texto_resp)

resultado = analizar_texto(TEXTO)
print(json.dumps(resultado, indent=2, ensure_ascii=False))